# Notebook 02: Manufacturing RAG Pipeline (Load, Split, Embed, Retrieve)

Demonstrates document ingestion from engineering markdown manuals, semantic text chunking, embedding generation, vector similarity indexing, and structured citation retrieval.


In [2]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

from backend.app.rag.loaders import ManufacturingDocLoader
from backend.app.rag.splitter import ManufacturingTextSplitter
from backend.app.rag.embeddings import FastDeterministicEmbeddings
from backend.app.rag.retriever import ManufacturingRetriever

# 1. Load documents
loader = ManufacturingDocLoader('../data/documents')
docs = loader.load()
print(f'Loaded {len(docs)} documents:')
for d in docs:
    print(f"- {d.metadata['source']} ({d.metadata['char_count']} chars)")


Loaded 6 documents:
- machine_manual.md (2227 chars)
- maintenance_sop.md (2596 chars)
- pressure_guidelines.md (1506 chars)
- safety_procedures.md (2152 chars)
- temperature_guidelines.md (1843 chars)
- vibration_guidelines.md (1854 chars)


## 2. Text Splitting and Metadata Preservation


In [3]:
splitter = ManufacturingTextSplitter(chunk_size=450, chunk_overlap=60)
chunks = splitter.split_documents(docs)
print(f'Generated {len(chunks)} semantic chunks.')
print('Sample chunk metadata:', chunks[0].metadata)
print('Sample chunk text:\n', chunks[0].page_content[:200])


Generated 44 semantic chunks.
Sample chunk metadata: {'source': 'machine_manual.md', 'file_path': '/Users/assujith/ManufacturingAgent/notebooks/../data/documents/machine_manual.md', 'title': 'Machine Manual', 'char_count': 2227, 'chunk_id': 0, 'section_title': 'Industrial Machine Specification Manual (Model: CNC-Mill-V4 / Lathe-Pro-X)', 'chunk_length': 76}
Sample chunk text:
 # Industrial Machine Specification Manual (Model: CNC-Mill-V4 / Lathe-Pro-X)


## 3. Querying the Retriever


In [4]:
retriever = ManufacturingRetriever(documents_dir='../data/documents', top_k=3)
query = 'ISO 10816 vibration velocity limits for spindle bearing fault'
evidence = retriever.retrieve(query, k=3)

print(f'Retrieved {len(evidence)} evidence chunks for: "{query}"\n')
for i, ev in enumerate(evidence):
    print(f"[{i+1}] Source: {ev['source']} (Section: {ev['section']}, Score: {ev['relevance']})")
    print(f"    {ev['content'][:180]}...\n")


Retrieved 3 evidence chunks for: "ISO 10816 vibration velocity limits for spindle bearing fault"

[1] Source: machine_manual.md (Section: 2. Standard Operating Parameters, Score: 0.355)
    ## 2. Standard Operating Parameters
- **Nominal Spindle Speed**: 1,200 to 8,500 RPM (Continuous Duty)
- **Maximum Spindle Speed**: 12,000 RPM (Intermittent Duty, max 15 minutes)
- ...

[2] Source: maintenance_sop.md (Section: 2. Maintenance Tiers and Frequency, Score: 0.3451)
    ## 2. Maintenance Tiers and Frequency
- **Daily Operator Check (Tier 1)**:
  - Visual inspection of hydraulic lines for weeping or pressure drops below 4.5 bar.
  - Spindle tempera...

[3] Source: vibration_guidelines.md (Section: 1. Scope & Vibration Measurement Principles, Score: 0.314)
    ## 1. Scope & Vibration Measurement Principles
Vibration telemetry is captured using triaxial piezoelectric accelerometers mounted directly to the spindle nose casting and main dri...

